## 1. Khai bao duong dan du lieu

In [1]:
from pathlib import Path

BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"

MAIN_DATA_PATH = DATA_DIR / "ai_company_adoption.csv"
# INDUSTRY_DATA_PATH = DATA_DIR / "ai_industry_summary.csv"
# COUNTRY_DATA_PATH = DATA_DIR / "country_ai_index.csv"

OUT_DIR = BASE_DIR / "processed_data"

## 2. Import thu vien can thiet

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import json

from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split

## 3. Doc cac bang du lieu CSV

In [3]:
df_main = pd.read_csv(MAIN_DATA_PATH)
# industry_df = pd.read_csv(INDUSTRY_DATA_PATH)
# country_df = pd.read_csv(COUNTRY_DATA_PATH)

## 4. Kiem tra gia tri thieu trong du lieu cong ty

In [4]:
df_main.isna().sum()

response_id                    0
company_id                     0
survey_year                    0
quarter                        0
country                        0
region                         0
industry                       0
company_size                   0
num_employees                  0
annual_revenue_usd_millions    0
company_founding_year          0
company_age                    0
company_age_group              0
ai_adoption_rate               0
ai_adoption_stage              0
years_using_ai                 0
ai_primary_tool                0
num_ai_tools_used              0
ai_use_case                    0
ai_projects_active             0
ai_training_hours              0
ai_budget_percentage           0
ai_maturity_score              0
ai_failure_rate                0
ai_investment_per_employee     0
regulatory_compliance_score    0
data_privacy_level             0
ai_ethics_committee            0
ai_risk_management_score       0
remote_work_percentage         0
employee_s

## 5. Kiem tra gia tri thieu trong du lieu nganh

In [5]:
# industry_df.isna().sum()

## 6. Kiem tra gia tri thieu trong du lieu quoc gia

In [6]:
# country_df.isna().sum()

## 7. Encoding du lieu phan loai (cyclical, ordinal, one-hot)

In [7]:
# Encoding day du tren bo du lieu goc ai_company_adoption.csv
raw_df = df_main.copy()

# Cac cot can loai bo theo yeu cau
drop_cols = ["response_id", "company_id", "survey_source", "data_collection_method", "company_age_group"]
raw_df = raw_df.drop(columns=drop_cols, errors="ignore")

identifier_cols = []
cyclical_cols = ["quarter"]
ordinal_categories = {
    "company_size": ["Startup", "SME", "Enterprise"],
    "company_age_group": ["0-5 years", "6-15 years", "16-30 years", "30+ years"],
    "ai_adoption_stage": ["none", "pilot", "partial", "full"],
    "data_privacy_level": ["Low", "Medium", "High"],
}

# 1) Binary columns (Yes/No)
binary_cols = []
for col in raw_df.select_dtypes(include=["object"]).columns:
    non_null_values = set(raw_df[col].dropna().unique().tolist())
    if non_null_values.issubset({"Yes", "No"}):
        binary_cols.append(col)

# 2) Nominal columns = object columns con lai sau khi loai tru cac nhom dac biet
object_cols = raw_df.select_dtypes(include=["object"]).columns.tolist()
nominal_cols = [
    col for col in object_cols
    if col not in set(identifier_cols + cyclical_cols + list(ordinal_categories.keys()) + binary_cols)
]

# 3) Tao bang mo ta chien luoc encoding cho tung cot
encoding_strategy = {}
for col in raw_df.columns:
    if col in identifier_cols:
        encoding_strategy[col] = "identifier_keep"
    elif col in cyclical_cols:
        encoding_strategy[col] = "cyclical_sin_cos"
    elif col in ordinal_categories:
        encoding_strategy[col] = "ordinal"
    elif col in binary_cols:
        encoding_strategy[col] = "binary_yes_no"
    elif col in nominal_cols:
        encoding_strategy[col] = "one_hot_k_minus_1"
    else:
        encoding_strategy[col] = "numeric_keep"

strategy_df = pd.DataFrame({
    "column": raw_df.columns,
    "strategy": [encoding_strategy[c] for c in raw_df.columns]
})

# 4) Thuc hien encoding
encoded_full_df = raw_df.copy()

# 4.1 Cyclical encoding cho quarter (Q1..Q4)
quarter_to_num = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4}
quarter_num = encoded_full_df["quarter"].astype(str).str.strip().str.upper().map(quarter_to_num)

unknown_quarter = encoded_full_df.loc[quarter_num.isna(), "quarter"].dropna().unique().tolist()
if unknown_quarter:
    raise ValueError(f"Gia tri quarter khong hop le: {unknown_quarter}")

encoded_full_df["quarter_sin"] = np.sin(2 * np.pi * (quarter_num - 1) / 4)
encoded_full_df["quarter_cos"] = np.cos(2 * np.pi * (quarter_num - 1) / 4)
encoded_full_df = encoded_full_df.drop(columns=["quarter"])

# 4.2 Ordinal encoding
for col, categories in ordinal_categories.items():
    if col not in encoded_full_df.columns:
        continue

    cat = pd.Categorical(encoded_full_df[col], categories=categories, ordered=True)
    unknown_values = encoded_full_df.loc[encoded_full_df[col].notna() & cat.isna(), col].unique().tolist()
    if unknown_values:
        raise ValueError(f"Gia tri khong nam trong thu bac cua cot {col}: {unknown_values}")

    encoded_full_df[col] = pd.Series(cat.codes, index=encoded_full_df.index).replace(-1, np.nan)

# 4.3 Binary encoding Yes/No
for col in binary_cols:
    encoded_full_df[col] = encoded_full_df[col].map({"No": 0, "Yes": 1})

# 4.4 One-hot encoding theo k-1 cot de tranh du bien
encoded_full_df = pd.get_dummies(
    encoded_full_df,
    columns=nominal_cols,
    drop_first=True,
    dtype=int
)

# Ban modeling (giong encoded_full_df vi da bo cot khong can thiet)
encoded_model_df = encoded_full_df.copy()

print("Cac cot da loai bo:", drop_cols)
print("Tong so cot goc:", df_main.shape[1])
print("Tong so cot sau khi bo cot:", raw_df.shape[1])
print("Tong so cot sau encoding:", encoded_model_df.shape[1])
print("Binary columns:", binary_cols)
print("One-hot columns (k-1):", nominal_cols)

strategy_df

Cac cot da loai bo: ['response_id', 'company_id', 'survey_source', 'data_collection_method', 'company_age_group']
Tong so cot goc: 43
Tong so cot sau khi bo cot: 38
Tong so cot sau encoding: 88
Binary columns: ['ai_ethics_committee']
One-hot columns (k-1): ['country', 'region', 'industry', 'ai_primary_tool', 'ai_use_case']


,column,strategy
0,survey_year,numeric_keep
1,quarter,cyclical_sin_cos
2,country,one_hot_k_minus_1
3,region,one_hot_k_minus_1
4,industry,one_hot_k_minus_1
5,company_size,ordinal
6,num_employees,numeric_keep
7,annual_revenue_usd_millions,numeric_keep
8,company_founding_year,numeric_keep
9,company_age,numeric_keep


# 8. Ghi dữ liệu đã được xử lý vào file CSV mới

In [8]:
encoded_model_df = encoded_model_df.drop(columns=['company_founding_year'], errors="ignore")

cols_to_move = ["cost_reduction_percent", "customer_satisfaction"]
existing_cols_to_move = [c for c in cols_to_move if c in encoded_model_df.columns]

encoded_model_df = encoded_model_df[
    [c for c in encoded_model_df.columns if c not in existing_cols_to_move] + existing_cols_to_move
]
OUT_DIR.mkdir(parents=True, exist_ok=True)
encoded_model_df.to_csv(OUT_DIR / "encoded_ai_company_adoption.csv", index=False)

In [9]:
# 1) Tách riêng cyclic_cols từ các cột có strategy = cyclical_sin_cos
cyclic_cols = []
for base_col, strategy in encoding_strategy.items():
    if strategy == "cyclical_sin_cos":
        for c in encoded_model_df.columns:
            if c == base_col or c.startswith(f"{base_col}_"):
                cyclic_cols.append(c)

# unique + giữ thứ tự
cyclic_cols = list(dict.fromkeys(cyclic_cols))

# 2) numeric_ordinal_cols = numeric_keep + ordinal (KHÔNG gồm cyclic)
numeric_ordinal_cols = [
    col for col, strategy in encoding_strategy.items()
    if strategy in {"numeric_keep", "ordinal"} and col in encoded_model_df.columns
]

# 3) onehot_cols = one-hot từ nominal + binary_yes_no
onehot_nominal_cols = [
    col for col in encoded_model_df.columns
    if any(col.startswith(f"{base_col}_") for base_col in nominal_cols)
]

binary_onehot_cols = [
    col for col, strategy in encoding_strategy.items()
    if strategy == "binary_yes_no" and col in encoded_model_df.columns
]

onehot_cols = list(dict.fromkeys(onehot_nominal_cols + binary_onehot_cols))

print(f"cyclic_cols: {len(cyclic_cols)}")
print(cyclic_cols)

print(f"\nnumeric_ordinal_cols: {len(numeric_ordinal_cols)}")
print(numeric_ordinal_cols)

print(f"\nonehot_cols: {len(onehot_cols)}")
print(onehot_cols)

print(f"\nbinary_onehot_cols: {len(binary_onehot_cols)}")
print(binary_onehot_cols)

meta_data = {
    "drop_cols": drop_cols,
    "encoding_strategy": encoding_strategy,
    "nominal_cols": nominal_cols,
    "cyclic_cols": cyclic_cols,
    "numeric_ordinal_cols": numeric_ordinal_cols,
    "onehot_cols": onehot_cols,
    "binary_cols": binary_onehot_cols,
    "counts": {
        "cyclic_cols": len(cyclic_cols),
        "numeric_ordinal_cols": len(numeric_ordinal_cols),
        "onehot_cols": len(onehot_cols),
        "binary_in_onehot_cols": len(binary_onehot_cols),
    },
}

OUT_DIR.mkdir(parents=True, exist_ok=True)
meta_path = OUT_DIR / "meta_data.json"

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta_data, f, ensure_ascii=False, indent=2)

print(f"\nSaved metadata to: {meta_path}")

cyclic_cols: 2
['quarter_sin', 'quarter_cos']

numeric_ordinal_cols: 30
['survey_year', 'company_size', 'num_employees', 'annual_revenue_usd_millions', 'company_age', 'ai_adoption_rate', 'ai_adoption_stage', 'years_using_ai', 'num_ai_tools_used', 'ai_projects_active', 'ai_training_hours', 'ai_budget_percentage', 'ai_maturity_score', 'ai_failure_rate', 'ai_investment_per_employee', 'regulatory_compliance_score', 'data_privacy_level', 'ai_risk_management_score', 'remote_work_percentage', 'employee_satisfaction_score', 'task_automation_rate', 'time_saved_per_week', 'productivity_change_percent', 'jobs_displaced', 'jobs_created', 'reskilled_employees', 'revenue_growth_percent', 'cost_reduction_percent', 'innovation_score', 'customer_satisfaction']

onehot_cols: 55
['country_Australia', 'country_Brazil', 'country_Canada', 'country_Chile', 'country_China', 'country_Colombia', 'country_Egypt', 'country_France', 'country_Germany', 'country_India', 'country_Indonesia', 'country_Italy', 'country